# 12 Causal Inference and Policy Evaluation

Does shower exposure really "cause" infection? Does disinfecting the water system really work?

Workflow: **DAG causal diagram → confounder/mediator/collider → attributable risk AR/PAR → DiD intervention evaluation → parallel trends check**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: DAG (directed acyclic graph) ---
# Describe causal relationships with text (no need to install graphviz)
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# DAG visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

# Nodes
nodes = {
    "floor/wing": (1, 5.5),
    "water\ncontamination": (3.5, 5.5),
    "shower\naerosol": (6, 5.5),
    "infection": (8.5, 5.5),
    "functional\nstatus": (1, 3),
    "shower\nuse": (4, 3),
    "age": (1, 1),
    "comorbidities": (3.5, 1),
    "severity": (6, 1),
    "death": (8.5, 1),
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Rectangle((x-0.7, y-0.4), 1.4, 0.8,
                 fill=True, facecolor="#e0e0e0", edgecolor="black", linewidth=1.5))
    ax.text(x, y, name, ha="center", va="center", fontsize=8, fontweight="bold")

# Arrows (causal direction)
arrows = [
    ("floor/wing", "water\ncontamination"),
    ("water\ncontamination", "shower\naerosol"),
    ("shower\naerosol", "infection"),
    ("functional\nstatus", "shower\nuse"),
    ("shower\nuse", "infection"),
    ("functional\nstatus", "infection"),
    ("age", "comorbidities"),
    ("comorbidities", "severity"),
    ("severity", "death"),
    ("infection", "severity"),
]

for src, dst in arrows:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.annotate("", xy=(x2-0.7, y2), xytext=(x1+0.7, y1),
                arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

ax.set_title("Legionella DAG — Causal Diagram", fontsize=14)
plt.tight_layout()
plt.show()

print("=== Identifying Causal Structures ===")
print("Confounder: functional_status → shower_use and → infection")
print("Mediator: shower_aerosol lies between water_contamination → infection")
print("Collider: hospitalized ← severity and ← infection")
print("\n→ Controlling for the confounder (done in Ch05) = correct")
print("→ Controlling for the collider = wrong! It creates a spurious association")

In [ ]:
# --- Step 2: Attributable Risk ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Attack rate for shower exposure
exposed = df[df["shower_use"] == 1]
unexposed = df[df["shower_use"] == 0]

risk_exposed = exposed["infected"].mean()
risk_unexposed = unexposed["infected"].mean()
risk_total = df["infected"].mean()

print("=== Attack Rate for Shower Exposure ===")
print(f"Attack rate among showerers: {risk_exposed:.1%} ({exposed['infected'].sum()}/{len(exposed)})")
print(f"Attack rate among non-showerers: {risk_unexposed:.1%} ({unexposed['infected'].sum()}/{len(unexposed)})")
print(f"Overall attack rate: {risk_total:.1%}")

# Attributable Risk
AR = risk_exposed - risk_unexposed
print(f"\n=== Attributable Risk (AR) ===")
print(f"AR = {risk_exposed:.3f} - {risk_unexposed:.3f} = {AR:.3f}")
print(f"→ Showerers have {AR:.1%} more infection risk than non-showerers")

# Population Attributable Risk
PAR = risk_total - risk_unexposed
PAR_pct = PAR / risk_total * 100
print(f"\n=== Population Attributable Risk (PAR) ===")
print(f"PAR = {risk_total:.3f} - {risk_unexposed:.3f} = {PAR:.3f}")
print(f"PAR% = {PAR_pct:.1f}%")
print(f"→ If shower exposure were eliminated, infections could theoretically drop by {PAR_pct:.0f}%")
print("→ Assumptions: the causal relationship holds, and there are no other transmission routes")

In [ ]:
# --- Step 3: Counterfactual Thinking ---
n_total = len(df)
n_infected = df["infected"].sum()

# Counterfactual: if no one had showered
counterfactual_cases = int(n_total * risk_unexposed)
prevented = n_infected - counterfactual_cases

print("=== Counterfactual Scenario ===")
print(f"Actual number infected: {n_infected}")
print(f"If no one had showered (counterfactual): {counterfactual_cases} expected infections")
print(f"Preventable infections: {prevented}")
print(f"\n→ But this is only a theoretical estimate!")
print("→ In practice, banning everyone from showering is not feasible")
print("→ A more realistic approach: disinfect the water system so showering becomes safe")

In [ ]:
# --- Step 4: DiD Data Preparation ---
# Scenario: on January 25, emergency water-system disinfection was carried out for Wing B on floors 2-3
# Treated group: Wing B on floors 2-3 (high attack rate, close to the contamination source)
# Control group: all of floor 1 + Wing A on floors 2-3 (different water supply or non-target area)

df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
cases = df[df["infected"] == 1].copy()

# Build daily panel data
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# Treated group: Wing B on floors 2-3
treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]
treated_daily = treated_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Control group: the remaining areas
control_cases = cases[~((cases["floor"].isin([2, 3])) & (cases["wing"] == "B"))]
control_daily = control_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Assemble into a panel
panel = pd.DataFrame({
    "date": list(all_dates) * 2,
    "treated": [1] * len(all_dates) + [0] * len(all_dates),
    "daily_cases": list(treated_daily.values) + list(control_daily.values),
})
panel["post"] = (panel["date"] >= "2026-01-25").astype(int)
panel["day"] = (panel["date"] - panel["date"].min()).dt.days

print("=== DiD Panel Data ===")
print(f"Treated group (Wing B, floors 2-3): {len(treated_daily)} days")
print(f"Control group (remaining areas): {len(control_daily)} days")
print(f"Intervention date: 2026-01-25")
print(f"\nCase counts before vs after intervention:")
summary = panel.groupby(["treated", "post"]).agg(total=('daily_cases','sum'), mean=('daily_cases','mean')).reset_index()
summary["group"] = summary["treated"].map({1: "Treated (Wing B, 2-3F)", 0: "Control"})
summary["period"] = summary["post"].map({0: "Before", 1: "After"})
print(summary[["group", "period", "total", "mean"]].to_string(index=False))

In [ ]:
# --- Step 5: Parallel Trends Check + DiD Visualization ---
fig, ax = plt.subplots(figsize=(10, 5))

# Treated group
ax.plot(all_dates, treated_daily.values, marker="o", markersize=4,
        label="Treated (Wing B, 2-3F)", color="#e34a33")
# Control group
ax.plot(all_dates, control_daily.values, marker="s", markersize=4,
        label="Control (rest)", color="#2c7fb8")

# Intervention line
ax.axvline(x=pd.Timestamp("2026-01-25"), color="black", linestyle="--",
           alpha=0.7, label="Intervention date (1/25)")

ax.set_title("DiD — Case Count Trends Before and After Intervention")
ax.set_xlabel("Date")
ax.set_ylabel("Daily case count")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("→ Check whether the two lines are roughly parallel before the intervention (left of the dashed line)")
print("→ If they are parallel, the DiD estimate is more trustworthy")

In [ ]:
# --- Step 6: DiD OLS Regression ---
model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()

print("=== DiD Regression Results ===")
print(model.summary().tables[1])

did_effect = model.params["treated:post"]
did_p = model.pvalues["treated:post"]

print(f"\n=== DiD Effect Estimate ===")
print(f"treated:post coefficient = {did_effect:.3f}")
print(f"p-value = {did_p:.4f}")

if did_effect < 0:
    print(f"\n→ After the intervention, the treated group had {abs(did_effect):.1f} fewer daily cases than expected")
else:
    print(f"\n→ After the intervention, the treated group had {did_effect:.1f} more daily cases than expected")

if did_p < 0.05:
    print("→ The effect is statistically significant (p < 0.05)")
else:
    print("→ The effect is not statistically significant (p ≥ 0.05)")
    print("→ Possible reasons: insufficient sample size, too short an observation window, or the effect needs more time to appear")

## Summary

| Step | Skill Learned |
|------|------------|
| DAG | Use diagrams to identify confounders, mediators, and colliders |
| AR / PAR | Quantify the proportion of infection attributable to an exposure |
| Counterfactual | Estimate the expected effect of "removing the exposure" |
| DiD | `cases ~ treated + post + treated:post` |
| Parallel trends | Whether the two groups' trends match before the intervention |

**Conclusions**:
- A DAG helps us clarify which variables to control for and which not to
- AR/PAR quantify an exposure's contribution, but only if the causal relationship holds
- DiD is a quasi-experimental method for evaluating intervention effects, but it requires the parallel trends assumption
- With observational data, causal inference always calls for caution

In the next chapter (Ch13), we make sure all analyses are reproducible → reproducible research.